# High dimensional experiment.

Dataset: EdNet 
Models: MoLA, NeuralCDM

Predictive performance is evaluated using negative log-likelihood (NLL), Brier score, and area under the receiver operating characteristic curve (AUC). NLL and Brier score assess the quality of predicted response probabilities, while AUC measures discrimination between correct and incorrect responses.

We additionally report computational characteristics relevant to high-dimensional application, including the number of learners, items, skills, and observed responses, as well as model training time and memory requirements where available. These measurements characterize the practical scalability of the competing approaches in a large educational response dataset.

The resulting evaluation therefore addresses three distinct questions: (1) whether each model can estimate its population-level parameters from a large response dataset, (2) whether those parameters support estimation of proficiency for previously unseen learners from an initial response history, and (3) whether the resulting learner representation provides useful predictions of subsequent item responses.

## Methodological Details

The empirical evaluation uses a learner-level partition to prevent responses from the same learner from contributing to both population-level parameter estimation and final evaluation. We first retain learners with at least 30 observed item responses. The eligible learners are randomly divided into a parameter-estimation set and an evaluation set, with 70% of eligible learners assigned to parameter estimation and 30% to evaluation. The learner-level partition is identical across MoLA and NeuralCDM.

For MoLA, the number of archetypes \(M\) is treated as a model-selection hyperparameter and is selected using only the parameter-estimation learners. Candidate values of \(M\) are evaluated using a held-out subset of the parameter-estimation learners, with predictive log likelihood used as the selection criterion. After selecting \(M\), the MoLA model is refit using all parameter-estimation learners with the selected number of archetypes. The resulting population-level parameters are then fixed for evaluation on the held-out evaluation learners. No responses from evaluation learners are used during selection of \(M\) or estimation of the final population-level parameters. NeuralCDM hyperparameters are selected analogously using only the parameter-estimation learners, with the same learner-level information boundary.


## Population-Level Parameter Estimation

For each model, all responses from learners in the parameter-estimation set are used to estimate the model's population-level parameters. For MoLA, these include the archetype proficiency parameters, item parameters, and mixture proportions. The resulting parameter estimates are denoted by $\hat{\Theta}_{\mathrm{MoLA}}$. NeuralCDM is trained using the same responses and the same item--skill associations available to MoLA.

No responses from learners in the evaluation set are used during this stage. After population-level estimation is complete, the resulting model parameters are fixed and are not subsequently updated using evaluation responses.

## Temporal Profiling of Evaluation Learners

For each evaluation learner $n$, responses are sorted according to their original chronological order. Let $T_n$ denote the learner's total number of eligible responses. The first 60\% of responses are assigned to a profiling set and the remaining 40\% are assigned to a prediction set. Let

$$
T_n^{\mathrm{profile}}
=
\left\lfloor 0.60T_n \right\rfloor
$$

denote the number of profiling responses. Because learners are required to have at least 30 responses, each learner contributes a minimum of 18 responses to the profiling period and at least 12 responses to the prediction period.

The profiling responses are used exclusively to estimate the learner-specific latent representation. The prediction responses are withheld from this estimation and are used only for evaluating subsequent response predictions.

## MoLA Learner Profiling

Given the population-level parameter estimates $\hat{\Theta}_{\mathrm{MoLA}}$, we estimate each evaluation learner's latent archetype responsibilities using only their profiling responses. Specifically, for learner $n$,

$$
\hat{\gamma}_{nm}
=
P(z_n=m
\mid
\mathbf{y}_{n,\mathrm{profile}},
\hat{\Theta}_{\mathrm{MoLA}}),
$$

where $z_n$ denotes the learner's latent archetype membership. The resulting responsibility vector $\hat{\boldsymbol{\gamma}}_n$ represents the learner's estimated position within the learned mixture of archetypes.

The population-level parameters remain fixed during this procedure. Thus, the profiling stage does not modify the learned archetypes, item parameters, or mixture proportions. Once $\hat{\boldsymbol{\gamma}}_n$ has been estimated, it is also held fixed throughout the prediction period.

For each item response $j$ in the prediction period, the predicted probability of a correct response is obtained by marginalizing over the learner's posterior archetype responsibilities,

$$
\hat{p}_{nj}
=
\sum_{m=1}^{M}
\hat{\gamma}_{nm}
P(Y_{nj}=1\mid z_n=m,\hat{\Theta}_{\mathrm{MoLA}}).
$$

Thus, prediction uses only the population-level parameters learned from the parameter-estimation learners and the evaluation learner's profiling history.

## NeuralCDM Learner Profiling

NeuralCDM is evaluated under the same information constraints. After training the population-level model using the parameter-estimation learners, its population-level parameters are held fixed. For each evaluation learner, the learner-specific proficiency representation is inferred using only the learner's profiling responses. The inferred representation is then held fixed when predicting responses in the subsequent prediction period.

No prediction-period responses are used to update either the learner representation or the global model parameters.

## Prediction Evaluation

Predictions are evaluated against the observed responses in the prediction period. For each learner, model performance is computed over responses that occur after the profiling period. We report negative log-likelihood (NLL), Brier score, and area under the receiver operating characteristic curve (AUC).

For binary responses $y_{nj}\in\{0,1\}$ and predicted probabilities $\hat{p}_{nj}$, the negative log-likelihood is

$$
\mathrm{NLL}
=
-\frac{1}{|\mathcal{D}_{\mathrm{pred}}|}
\sum_{(n,j)\in\mathcal{D}_{\mathrm{pred}}}
\left[
y_{nj}\log\hat{p}_{nj}
+
(1-y_{nj})\log(1-\hat{p}
$$
